# Plot all PACE from eddy of interest

Authors: SEL, LJK 

In [1]:
import earthaccess
import xarray as xr
from xarray.backends.api import open_datatree
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import matplotlib.animation as animation
import os

In [2]:
auth = earthaccess.login(persist=True)

### TO DO: Switch out specialid for correct eddy ID based on AVISO eddy product. Switch out AVISO eddy filename below for correct one

In [3]:
aviso_ds = xr.open_dataset('data/eddy_trajectories/Edward_Eddy_trajectory_nrt_3.2exp_cyclonic_20180101_20240723.nc')
aviso_ds
specialid='eddy_IDedward'

### Select directory where you want images to be saved

In [4]:
image_dir = '/home/jovyan/go-swace/figures/edward/pace/'
specialpath=image_dir+specialid

In [5]:
dates = [i for i in aviso_ds.time.values] # these are sorted already
print(min(dates))
print(max(dates)) # still existed at the end of the dataset
center_lats = [i for i in aviso_ds.latitude.values]
center_lons = [i for i in aviso_ds.longitude.values]
print(len(center_lats))

2023-12-14T00:00:00.000000000
2024-07-23T00:00:00.000000000
223


In [6]:
tminny=str(min(dates))
tminny=tminny[0:10]
tmaxxy=str(max(dates))
tmaxxy=tmaxxy[0:10]
tmin,tmax = tminny,tmaxxy

In [7]:
center_lats = [i for i in aviso_ds.latitude.values]
center_lons = [i for i in aviso_ds.longitude.values]
print(len(center_lats))

223


In [8]:
#Define box based on eddy
latmin,latmax = min(center_lats)-2,max(center_lats)+2
lonmin,lonmax = min(center_lons)-360-2,max(center_lons)-360+2

In [9]:
tmax

'2024-07-23'

In [10]:
# tspan = ("2024-04-01", "2024-09-20")
tspan=(tmin,tmax)
clouds = (0, 100)
bbox = (lonmin, latmin, lonmax, latmax) #edward

In [11]:
results = earthaccess.search_data(
    short_name="PACE_OCI_L2_BGC",
    temporal=tspan,
    bounding_box=bbox,
    cloud_cover=clouds,
)

In [12]:
paths = earthaccess.open(results) #'streaming' data

QUEUEING TASKS | :   0%|          | 0/240 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/240 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/240 [00:00<?, ?it/s]

In [13]:
def get_eddy_by_ID_date(ds,eddy_date):
    """
    ds: netCDF AVISO format
    track_id: id of eddy to extract
    eddy_date: date in format 'YYYY-MM-DD'

    Returns contour lons, contour lats, center lon, center lat
    """
    try:
        ind = np.where(ds.time == np.datetime64(eddy_date))[0][0]
        contour_lons = np.array(ds.effective_contour_longitude[ind])
        contour_lats = np.array(ds.effective_contour_latitude[ind])
    except:
        print('No eddy data available with that request ... :(')        
    
    return contour_lons,contour_lats,ds.longitude[ind],ds.latitude[ind]

In [14]:
len(paths)

240

In [17]:
for index in range(0,len(paths)):
    datatree = open_datatree(paths[index])
    datatree
    dataset = xr.merge(datatree.to_dict().values())
    dataset
    dataset = dataset.set_coords(("longitude", "latitude"))

    h=str(paths[index])
    datey=h[64:72]
    
    date_requested=datey
    contour_lons,contour_lats,center_lon,center_lat = get_eddy_by_ID_date(aviso_ds,'%s-%s-%s'%(date_requested[0:4],date_requested[4:6],date_requested[6:8]))

    fig = plt.figure()
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.coastlines()
    ax.gridlines(draw_labels={"left": "y", "bottom": "x"})
    plot = dataset["chlor_a"].plot(x="longitude", y="latitude", cmap="Spectral_r", vmax=0.5, ax=ax)
    ax.set_ylabel('latitude')
    ax.set_xlabel('longitude')
    ax.set_ylim([latmin,latmax])
    ax.set_xlim([lonmin,lonmax])
    ax.plot(contour_lons-360,contour_lats,zorder=100,c='k',linewidth=2)
    ax.set_title(datey[0:4]+'-'+datey[4:6]+'-'+datey[6:len(datey)])
    if len(str(index))==1:
        plt.savefig(str(specialpath)+'CHL'+str(0)+str(0)+str(index).split('.')[0]+'pace.png')
    elif len(str(index))==2:
        plt.savefig(str(specialpath)+'CHL'+str(0)+str(index).split('.')[0]+'pace.png')
    else:
        plt.savefig(str(specialpath)+'CHL'+str(index).split('.')[0]+'pace.png')
    plt.close()

In [18]:
# Get a list of all PNG files in the directory
images = [img for img in os.listdir(image_dir) if img.endswith("pace.png")]
images.sort()  # Sort the images if needed
# Create a figure and axis for the animation
fig, ax = plt.subplots()

# Function to update the figure for each frame
def update(frame):
    img_path = os.path.join(image_dir, images[frame])
    img = plt.imread(img_path)
    ax.imshow(img)
    ax.axis('off')  # Hide axes
# Create the animation
ani = animation.FuncAnimation(fig, update, frames=len(images), repeat=True)

# Save the animation as a GIF or display it
# ani.save('animation_edward_all.gif', writer='imagemagick', fps=3)  # Save as GIF #was fps=2
ani.save('animation_PACE_' + str(specialid) + '.gif', writer='imagemagick', fps=4)  # fps is frames per second. Lower numbers = slower speeds

# plt.show()  # Uncomment to display the animation

print("Animation created successfully.")